# P08 — deadline-day preliminary pilot

Select **T4 GPU**, run setup cells 1–2, then the smoke check if not already verified. The pilot cell defaults to zero-shot plus the three matched-budget placements, seed 42 only. It uses a fixed 1,024-example training subset, 32 optimizer updates, and the same 64 validation examples for every configuration. These are **single-seed, short-training, subset-validation results**, not full benchmarks or baseline reproduction. No multi-seed uncertainty can be inferred.

If time remains, extend `PILOT_MODES` with all-attention LoRA, full fine-tuning, and Prefix Tuning. Full three-seed experiments, rank/projection ablations and the human study remain outstanding. The provided rubric has not been waived. Do not run the original long research cell for this pilot.


In [ ]:
# 1. Mount persistent storage and select the study directory.
from google.colab import drive
from pathlib import Path
import json, os, subprocess, sys

drive.mount('/content/drive')
STUDY = 'midterm_fp16_v1'  # Use a new name for a different experimental protocol.
ROOT = Path('/content/drive/MyDrive/CISC473_P08') / STUDY
ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
print('Persistent results:', ROOT)


In [ ]:
# 2. Install the tested code into an isolated environment (no kernel restart required).
CODE_REVISION = '6c10d0a09b9a4faa5eccf4cbb32276eb640091f0'  # Merged, tested experiment implementation.
MODEL_REVISION = 'aadd2ab0ae0c8268c7c9693540e9904811f36177'
CNN_REVISION = '96df5e686bee6baa90b8bee7c28b81fa3fa6223d'
XSUM_REVISION = '7d4d486c2f8ef850b1a11aead99b894ff3dd7da9'
REPO = Path('/content/DeepLearningP08')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/wenbin05/DeepLearningP08.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', CODE_REVISION], check=True)
# Colab's notebook kernel may use Python 3.13+. Install a separate pinned
# interpreter for this project's dependencies; leave the notebook kernel alone.
PYTHON_VERSION = '3.12.12'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv==0.9.4'], check=True)
BOOTSTRAP_ENV = dict(os.environ, UV_NATIVE_TLS='true')
subprocess.run([sys.executable, '-m', 'uv', 'python', 'install', PYTHON_VERSION],
               env=BOOTSTRAP_ENV, check=True)
VENV = Path('/content/p08-py312')
if not (VENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', PYTHON_VERSION,
                    '--seed', str(VENV)], env=BOOTSTRAP_ENV, check=True)
PYTHON = str(VENV / 'bin/python')
project_version = subprocess.check_output(
    [PYTHON, '-c', 'import platform; print(platform.python_version())'], text=True).strip()
if project_version != PYTHON_VERSION:
    raise RuntimeError(f'Expected project Python {PYTHON_VERSION}, got {project_version}. Use a fresh runtime.')
print('Colab kernel:', sys.version.split()[0], '| Project Python:', project_version)
LOCK = ROOT / 'environment-lock.txt'
requirements = LOCK if LOCK.exists() else REPO / 'requirements.txt'
subprocess.run([PYTHON, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
if not LOCK.exists():
    LOCK.write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze', '--all'], text=True))
subprocess.run([PYTHON, '-c', 'import torch; print(torch.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"); assert torch.cuda.is_available(), "Select a GPU runtime"'], check=True)
ENV = dict(os.environ, PYTHONHASHSEED='0', OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')

# Refuse to combine studies with different declared protocols.
gpu_name = subprocess.check_output([PYTHON, '-c', 'import torch; print(torch.cuda.get_device_name())'], text=True).strip()
protocol = dict(code=CODE_REVISION, model=MODEL_REVISION, cnn=CNN_REVISION,
                xsum=XSUM_REVISION, precision='fp16', train_samples=10000,
                epochs=3, warmup_steps=500, data_seed=42,
                python=subprocess.check_output([PYTHON, '-c', 'import sys; print(sys.version)'], text=True).strip(),
                gpu=gpu_name)
protocol_path = ROOT / 'protocol.json'
if protocol_path.exists() and json.loads(protocol_path.read_text()) != protocol:
    raise RuntimeError('Study protocol changed: choose a new STUDY directory.')
protocol_path.write_text(json.dumps(protocol, indent=2) + '\n')

# Stream subprocess output to the cell and a persistent log; propagate every failure.
def run(arguments, log_name):
    logs = ROOT / 'logs'
    logs.mkdir(exist_ok=True)
    with (logs / log_name).open('a') as log:
        process = subprocess.Popen([PYTHON, '-u', *map(str, arguments)], cwd=REPO,
            env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in process.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            if process.wait():
                raise RuntimeError(f'Command failed. See {logs / log_name}')
        except BaseException:
            process.terminate()
            process.wait()
            raise


In [ ]:
# 3. Real-model smoke check. Outputs here are NOT research results.
# Change SMOKE_ID only if you deliberately want a fresh execution check.
SMOKE_ID = 'check_1'
smoke_root = ROOT / 'smoke' / SMOKE_ID
smoke_run = smoke_root / 'cross_attn_r8_q_proj,v_proj_seed42'
if (smoke_run / 'validation_results.json').exists():
    print('Smoke already completed:', smoke_run)
elif smoke_run.exists():
    raise RuntimeError('Smoke run was interrupted. Set a new SMOKE_ID to retry; existing files are preserved.')
else:
    run(['-m', 'src.train', '--placement', 'cross_attn', '--seed', '42',
         '--max_train_samples', '8', '--max_eval_samples', '2', '--max_steps', '1',
         '--warmup_steps', '0', '--fp16', '--model_revision', MODEL_REVISION,
         '--dataset_revision', CNN_REVISION, '--output_dir', smoke_root], 'smoke.log')
print(json.loads((smoke_run / 'validation_results.json').read_text())['metrics'])


## Short pilot — run this instead of the full research matrix
The first priority is genuine, comparable evidence. All three isolated placements keep rank 8, Q/V targets and 147,456 trainable parameters. Data membership, preprocessing, updates, warmup, generation settings and seed remain fixed. Short training may not converge; do not interpret a ranking as a reliable final conclusion.


In [ ]:
# 4. Bounded pilot. Existing full-run outputs are preserved in their original folder.
PILOT_MODES = ['zero_shot', 'encoder_self_attn', 'decoder_self_attn', 'cross_attn']
# If time permits, rerun this cell with the extra baselines appended:
# PILOT_MODES += ['all_attn', 'full_ft', 'prefix']
PILOT = ROOT.parent / 'pilot_seed42_train1024_val64_steps32_v1'
PILOT.mkdir(parents=True, exist_ok=True)
pilot_protocol = dict(protocol, train_samples=1024, validation_samples=64,
                      max_steps=32, epochs=1, warmup_steps=4, training_seed=42,
                      purpose='single-seed short-training validation pilot')
pilot_protocol_path = PILOT / 'pilot_protocol.json'
if pilot_protocol_path.exists() and json.loads(pilot_protocol_path.read_text()) != pilot_protocol:
    raise RuntimeError('Existing pilot protocol differs; use a new PILOT directory.')
pilot_protocol_path.write_text(json.dumps(pilot_protocol, indent=2) + '\n')
(PILOT / 'environment-lock.txt').write_text(LOCK.read_text())
for mode in PILOT_MODES:
    directory = PILOT / f'{mode}_r8_q_proj,v_proj_seed42'
    completed = directory / 'validation_results.json'
    if completed.exists():
        saved = json.loads(completed.read_text())
        manifest = saved['manifest']
        config = manifest['config']
        val = manifest['data']['validation']
        train = manifest['data']['train']
        expected = dict(placement=mode, model_revision=MODEL_REVISION,
                        dataset_revision=CNN_REVISION, lora_rank=8,
                        projections='q_proj,v_proj', fp16=True, warmup_steps=4,
                        num_train_epochs=1, data_seed=42)
        if saved.get('status') != 'completed' or manifest['seed'] != 42 or any(config[k] != v for k,v in expected.items()) or val['samples'] != 64 or manifest['training_arguments']['max_steps'] != 32 or (mode != 'zero_shot' and train['samples'] != 1024):
            raise RuntimeError(f'Unexpected saved pilot protocol: {directory}')
        print('Completed previously:', mode)
    elif directory.exists():
        raise RuntimeError(f'Incomplete run preserved: {directory}. Move it outside the pilot folder before retrying this mode.')
    else:
        run(['-m', 'src.train', '--placement', mode, '--seed', '42', '--fp16',
             '--max_train_samples', '1024', '--max_eval_samples', '64',
             '--max_steps', '32', '--num_train_epochs', '1', '--warmup_steps', '4',
             '--model_revision', MODEL_REVISION, '--dataset_revision', CNN_REVISION,
             '--output_dir', PILOT], f'pilot_{mode}_seed42.log')
print('Pilot files:', PILOT)


In [ ]:
# 5. Generate a report table from completed genuine outputs, even if only some runs finished.
# This table reports VALIDATION SUBSET scores, not full CNN/DailyMail test scores.
from IPython.display import Markdown, display
rows = []
records = []
for path in sorted(PILOT.glob('*/validation_results.json')):
    result = json.loads(path.read_text())
    if result.get('status') != 'completed':
        continue
    m = result['manifest']
    scores = result['metrics']
    if m['seed'] != 42 or m['data']['validation']['samples'] != 64 or m['training_arguments']['max_steps'] != 32:
        raise RuntimeError(f'Not a matching pilot result: {path}')
    if m['config']['placement'] != 'zero_shot' and m['data']['train']['samples'] != 1024:
        raise RuntimeError(f'Different training subset size: {path}')
    rows.append('| {} | {:,} | {:.2f} | {:.2f} | {:.2f} | {:.4f} | {:.1f} |'.format(
        m['config']['placement'], m['parameters']['trainable'],
        scores['validation_rouge1'], scores['validation_rouge2'], scores['validation_rougeL'],
        scores['validation_loss'], result['training_seconds']))
    records.append(dict(source_file=str(path), result=result))
if not rows:
    raise RuntimeError('No completed pilot outputs yet. Do not invent missing entries.')
caption = ('Single-seed pilot (seed 42): fixed 1,024-example CNN/DailyMail training subset; '
           '32 optimizer updates for trained models; 64-example validation subset; '
           'official stemmed ROUGE F1 ×100 against full references. No standard deviations '
           'are available. Short training is not evidence of convergence or published-baseline reproduction.')
table = ('| Configuration | Trainable parameters | ROUGE-1 | ROUGE-2 | ROUGE-L | Validation loss | Training seconds |\n'
         '|---|---:|---:|---:|---:|---:|---:|\n' + '\n'.join(rows) + '\n\n' + caption)
(PILOT / 'preliminary_table.md').write_text(table + '\n')
(PILOT / 'completed_pilot_outputs.json').write_text(json.dumps(records, indent=2) + '\n')
display(Markdown(table))
print('For report revision, share preliminary_table.md and completed_pilot_outputs.json from:', PILOT)


## Optional: small held-out transfer check, only if time remains
Run the next cell after preserving the preliminary table. Each domain uses a fixed **64-example test subset**, with no extra training. These are subset test results; label the domain, split, sample size and single seed. They do not replace full test-set, three-seed evaluations.


In [ ]:
# 6. OPTIONAL — adds CNN/DailyMail and XSum held-out subset evaluations.
for directory in sorted(PILOT.glob('*_r8_q_proj,v_proj_seed42')):
    if not (directory / 'validation_results.json').exists():
        continue
    for dataset, revision in [('cnn_dailymail', CNN_REVISION), ('xsum', XSUM_REVISION)]:
        files = list((directory / 'evaluations').glob(f'{dataset}_*/test_results.json'))
        if files:
            if len(files) != 1:
                raise RuntimeError(f'Multiple evaluations require inspection: {directory}')
            saved = json.loads(files[0].read_text())
            meta = saved['manifest']['evaluation']
            if saved.get('status') != 'completed' or meta['samples'] != 64 or meta['revision'] != revision:
                raise RuntimeError(f'Different saved test protocol: {files[0]}')
            print('Subset test already complete:', files[0])
            continue
        run(['-m', 'src.train', '--eval_only', directory, '--eval_dataset', dataset,
             '--eval_revision', revision, '--max_eval_samples', '64'],
            f'pilot_{directory.name}_{dataset}.log')


## Report wording and remaining work
Describe every completed row as a preliminary single-seed pilot. Do not use “mean ± standard deviation,” “full test set,” “published baseline reproduced,” or a definitive placement-superiority claim for this pilot. List full fine-tuning, Prefix Tuning, all-attention LoRA, additional seeds, full test/transfer evaluations, rank/projection ablations and the human study as outstanding unless their genuine outputs exist. Explain the deadline/compute limitation and give a concrete continuation plan. Functional code and a pilot do not guarantee full marks for the rubric's reproduction requirement.
